# 10.3 · 经典 CNN 架构 / Classic CNN Architectures

> **课程定位 / Where this fits**
> 第 3 课，**Part 10 · 计算机视觉**。
> Lesson 3, **Part 10 · Computer Vision**.
>
> 10.2 我们搭了个小 CNN。这一课沿着历史看 **CNN 架构如何一步步进化**：从 1998 年的 **LeNet**，到 2012 年点燃深度学习的 **AlexNet**，到极简堆叠的 **VGG**，再到用 Inception/1×1 卷积的 **GoogLeNet**。每一代都带来一两个关键设计——理解这些设计(而非死记网络层数)正是面试想考的。我们会**亲手搭 LeNet 训练**，并用小实验验证"两个 3×3 = 一个 5×5""1×1 卷积省参数"等核心思想。
> In 10.2 we built a small CNN. This lesson traces **how CNN architectures evolved**: **LeNet** (1998), **AlexNet** (2012, igniting deep learning), minimalist **VGG**, and **GoogLeNet** with Inception/1×1 convolutions. Each generation brought a key design — understanding these (not memorizing layer counts) is what interviews probe. We'll **build and train LeNet** and run small experiments verifying "two 3×3 = one 5×5" and "1×1 conv saves params."
>
> 💼 **实战/面试视角**："为什么用 3×3 小卷积堆叠 / 1×1 卷积干什么 / VGG vs GoogLeNet" 高频。
> 💼 **Practical/interview angle:** "why stack small 3×3 convs / what 1×1 conv does / VGG vs GoogLeNet" — frequent.

> 📐 **符号约定 / Notation**
> - top-5 error —— ImageNet 上预测前5含正确答案的错误率(越低越好) / ImageNet top-5 error
> - 感受野 —— 见 10.2 / receptive field, see 10.2

> 💡 **面试相关 / Interview-relevant**
> - "为什么用多个 3×3 代替大卷积核"（出镜率 ★★★★★）
> - "1×1 卷积的作用"（★★★★★，降维/升维/跨通道融合）
> - "AlexNet 相比 LeNet 的关键改进"（★★★，ReLU/Dropout/更深/GPU）
> - "Inception 模块的思想"（★★★，多尺度并行）

---

## 学习目标 / Learning Objectives
1. 了解 CNN 架构演进主线及每代的关键贡献。
   Know the CNN evolution and each generation's key contribution.
2. 亲手搭建并训练 **LeNet-5**。
   Build and train LeNet-5 by hand.
3. 理解 **VGG**：为什么用多个 3×3 堆叠（感受野+参数+非线性）。
   Understand VGG: why stack 3×3 (receptive field + params + nonlinearity).
4. 理解 **1×1 卷积** 与 **Inception** 模块。
   Understand 1×1 convolutions and the Inception module.

## 目录 / TOC
1. [架构演进史（可视化）⭐](#1)
2. [LeNet-5：搭建并训练 ⭐](#2)
3. [VGG：为什么用 3×3 堆叠 ⭐](#3)
4. [1×1 卷积与 Inception + 小结 ⭐](#4)


<a id="1"></a>
## 1. 架构演进史（可视化）⭐ / Architecture Evolution

CNN 架构的发展几乎都由 **ImageNet 图像分类竞赛**推动（120 万张图，1000 类）。主线：
CNN architecture progress was largely driven by the **ImageNet classification challenge** (1.2M images, 1000 classes). The main line:
- **LeNet-5 (1998, LeCun)**：第一个成功的 CNN，识别手写数字。卷积+池化+全连接的范式从此确立。
  **LeNet-5 (1998, LeCun):** the first successful CNN, for handwritten digits. Established the conv+pool+FC paradigm.
- **AlexNet (2012)**：在 ImageNet 上碾压传统方法，**点燃了整个深度学习浪潮**。关键：更深、**ReLU**、**Dropout**、数据增强、用 **GPU** 训练。
  **AlexNet (2012):** crushed classical methods on ImageNet, **igniting the deep-learning wave**. Keys: deeper, **ReLU**, **Dropout**, augmentation, **GPU** training.
- **VGG (2014)**：极简哲学——**只用 3×3 卷积**，靠堆叠加深到 16/19 层。证明"深度"很重要。
  **VGG (2014):** minimalist — **only 3×3 convs**, stacked to 16/19 layers. Showed depth matters.
- **GoogLeNet/Inception (2014)**：**Inception 模块**（多尺度并行）+ **1×1 卷积**降维，用更少参数达到更高精度。
  **GoogLeNet/Inception (2014):** **Inception modules** (multi-scale parallel) + **1×1 convs** for dimensionality reduction; higher accuracy with fewer params.
- **ResNet (2015)**：**残差连接**让网络能堆到 100+ 层（下一课专讲）。
  **ResNet (2015):** **residual connections** enabled 100+ layers (next lesson).

下面把它们的 ImageNet top-5 错误率画出来——**越来越低**，正是架构进步的缩影。
Let's plot their ImageNet top-5 error — **dropping steadily**, the epitome of architectural progress.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn
sns.set_theme(style="whitegrid")

# ImageNet top-5 错误率(公开数值, 近似) / well-known ImageNet top-5 error rates
models = ["传统方法\n(2011)", "AlexNet\n(2012)", "VGG\n(2014)", "GoogLeNet\n(2014)", "ResNet\n(2015)", "人类\n(~5%)"]
top5_err = [25.8, 16.4, 7.3, 6.7, 3.6, 5.1]
colors = ["#999","#e67","#5a9","#5a9","#39c","#fc3"]
fig, ax = plt.subplots(figsize=(9,4))
bars = ax.bar(models, top5_err, color=colors)
for b, e in zip(bars, top5_err): ax.text(b.get_x()+b.get_width()/2, e+0.4, f"{e}%", ha="center", fontsize=9)
ax.axhline(5.1, color="#fc3", ls="--", lw=1)
ax.set_ylabel("ImageNet top-5 错误率 (%)"); ax.set_title("CNN 架构进步: top-5 错误率逐年下降, 2015 起超越人类")
plt.tight_layout(); plt.show()
print("2012 AlexNet 大幅领先传统方法 → 点燃深度学习; 2015 ResNet 超越人类水平(~5%)")
print("演进关键词: 更深 + ReLU/Dropout(AlexNet) → 3×3堆叠(VGG) → 1×1/Inception(GoogLeNet) → 残差(ResNet)")


<a id="2"></a>
## 2. LeNet-5：搭建并训练 ⭐ / LeNet-5: Build & Train

**LeNet-5** 是 CNN 的"始祖"，结构非常清晰：**两组(卷积+池化) → 三层全连接**。当年用于识别支票上的手写数字。我们用现代写法（ReLU 代替原版 tanh、max-pool 代替原版 avg-pool）在 FashionMNIST 上复现并训练。
**LeNet-5** is the CNN ancestor with a very clear structure: **two (conv+pool) blocks → three FC layers**. Originally for handwritten digits on checks. We reproduce it with modern touches (ReLU instead of tanh, max-pool instead of avg-pool) and train on FashionMNIST.


In [ ]:
import os, torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
tfm = transforms.ToTensor()
train_full = torchvision.datasets.FashionMNIST(DATA_ROOT, train=True, download=True, transform=tfm)
test_full  = torchvision.datasets.FashionMNIST(DATA_ROOT, train=False, download=True, transform=tfm)
classes = train_full.classes
train_loader = DataLoader(Subset(train_full, range(12000)), batch_size=128, shuffle=True)
test_loader  = DataLoader(Subset(test_full, range(2000)), batch_size=256)

# LeNet-5 (现代版): 2×(conv+pool) → 3×FC / modern LeNet-5
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),   # (1,28,28)→(6,14,14)
            nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2))             # →(16,5,5)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(16*5*5, 120), nn.ReLU(),            # 全连接 120 / FC 120
            nn.Linear(120, 84), nn.ReLU(), nn.Linear(84, 10))          # 84 → 10 类 / 84 → 10 classes
    def forward(self, x): return self.classifier(self.features(x))

torch.manual_seed(0); net = LeNet(); opt = torch.optim.Adam(net.parameters(), lr=1e-3); ce = nn.CrossEntropyLoss()
loss_hist, acc_hist = [], []
for epoch in range(6):
    net.train()
    for xb, yb in train_loader:
        opt.zero_grad(); loss = ce(net(xb), yb); loss.backward(); opt.step()
    net.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            correct += (net(xb).argmax(1)==yb).sum().item(); total += len(yb)
    loss_hist.append(loss.item()); acc_hist.append(correct/total)
print(f"LeNet 参数量 = {sum(p.numel() for p in net.parameters()):,}")
print(f"最终 test 准确率 = {acc_hist[-1]:.3f}")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.plot(loss_hist, "o-"); a1.set_xlabel("epoch"); a1.set_ylabel("训练损失"); a1.set_title("LeNet 训练损失")
a2.plot(acc_hist, "o-", color="#39c"); a2.set_xlabel("epoch"); a2.set_ylabel("test 准确率"); a2.set_title("LeNet 测试准确率")
plt.tight_layout(); plt.show()
print("LeNet 范式: (卷积提取特征+池化降维)×N → 全连接分类; 至今仍是 CNN 的基本骨架")


<a id="3"></a>
## 3. VGG：为什么用 3×3 堆叠 ⭐ / VGG: Why Stack 3×3

VGG 的核心主张：**与其用一个大卷积核(如 5×5、7×7)，不如堆叠多个 3×3 小核**。这是面试超高频题，原因有三：
VGG's core claim: **instead of one large kernel (5×5, 7×7), stack several small 3×3 kernels**. A super-frequent interview question; three reasons:
1. **相同感受野**：两个 3×3 堆叠的感受野 = 一个 5×5；三个 3×3 = 一个 7×7。
   **Same receptive field:** two stacked 3×3 = one 5×5; three 3×3 = one 7×7.
2. **更少参数**：两个 3×3 是 $2×(3×3)=18$ 个权重/通道，一个 5×5 是 $25$ 个——更省。
   **Fewer params:** two 3×3 = $2×9=18$ weights/channel vs one 5×5 = $25$ — cheaper.
3. **更多非线性**：每个 3×3 后都接一个 ReLU，堆叠 = 更多非线性变换 = 更强表达力。
   **More nonlinearity:** a ReLU after each 3×3, so stacking = more nonlinear transforms = more expressive.

下面用代码验证"感受野相同、参数更少"。
Let's verify "same receptive field, fewer params" in code.


In [ ]:
# 验证1: 两个3×3 与 一个5×5 的感受野相同 / two 3x3 == one 5x5 receptive field
two_3x3 = nn.Sequential(nn.Conv2d(8, 8, 3, padding=0), nn.Conv2d(8, 8, 3, padding=0))
one_5x5 = nn.Conv2d(8, 8, 5, padding=0)
x = torch.randn(1, 8, 10, 10)
print(f"输入 10×10 → 两个3×3: 输出 {tuple(two_3x3(x).shape[2:])}  (10→8→6)")
print(f"输入 10×10 → 一个5×5: 输出 {tuple(one_5x5(x).shape[2:])}  (10→6)")
print("两者输出尺寸相同 → 感受野等价(都'看'5×5区域)\n")

# 验证2: 参数量对比 (in=out=64 通道) / param comparison
C = 64
p_two = 2 * (3*3*C*C)        # 两个3×3卷积 / two 3x3 convs
p_one = (5*5*C*C)            # 一个5×5卷积 / one 5x5 conv
print(f"通道数 {C} 时:")
print(f"  两个3×3: {p_two:,} 参数")
print(f"  一个5×5: {p_one:,} 参数  ← 多了 {(p_one-p_two)/p_two*100:.0f}%")
print("结论: 多个小3×3 = 相同感受野 + 更少参数 + 更多非线性(ReLU) → VGG 的设计精髓")

fig, ax = plt.subplots(figsize=(5.5,3.6))
ax.bar(["两个 3×3","一个 5×5"], [p_two, p_one], color=["#5a9","#e67"])
ax.set_ylabel("参数量"); ax.set_title("相同感受野下: 3×3堆叠更省参数")
for i,v in enumerate([p_two,p_one]): ax.text(i, v+1500, f"{v:,}", ha="center")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. 1×1 卷积与 Inception + 小结 ⭐ / 1×1 Conv & Inception

**1×1 卷积**听起来很奇怪——一个只看单个像素的核有什么用？关键在**通道维度**：1×1 卷积不混合空间信息，但**对每个像素位置的所有通道做加权组合**，因此可以：
A **1×1 convolution** sounds odd — what use is a kernel seeing one pixel? The point is the **channel dimension**: a 1×1 conv doesn't mix spatial info but **linearly combines all channels at each pixel**, so it can:
- **降维/升维**：把通道数从 256 压到 64（或反之），**大幅减少后续计算**。GoogLeNet 用它在昂贵的大卷积前先降通道。
  **Reduce/expand channels:** squeeze 256→64 (or vice versa), **slashing downstream compute**. GoogLeNet uses it to cut channels before expensive convs.
- **跨通道信息融合 + 增加非线性**（配 ReLU）。
  **Cross-channel mixing + extra nonlinearity** (with ReLU).

**Inception 模块**：不纠结"该用 1×1 还是 3×3 还是 5×5"，干脆**并行全用**，再把结果拼接——让网络自己学多尺度特征。1×1 卷积在其中负责降维控制计算量。
**Inception module:** instead of choosing 1×1 vs 3×3 vs 5×5, **run them all in parallel** and concatenate — letting the net learn multi-scale features. 1×1 convs keep compute in check.


In [ ]:
# 1×1 卷积降维省计算 / 1x1 conv reduces channels and compute
x = torch.randn(1, 256, 28, 28)                       # 256 通道特征图 / 256-channel map
# 直接用 5×5 卷积(256→256): 很贵 / direct 5x5 conv: expensive
direct = nn.Conv2d(256, 256, 5, padding=2)
# 先 1×1 降到 64, 再 5×5, 再 1×1 升回 256: 便宜很多 / bottleneck via 1x1
bottleneck = nn.Sequential(nn.Conv2d(256,64,1), nn.ReLU(), nn.Conv2d(64,64,5,padding=2),
                           nn.ReLU(), nn.Conv2d(64,256,1))
def params(m): return sum(p.numel() for p in m.parameters())
print(f"直接 5×5 (256→256):        {params(direct):,} 参数")
print(f"1×1降维瓶颈 (256→64→64→256): {params(bottleneck):,} 参数  ← 省 {(1-params(bottleneck)/params(direct))*100:.0f}%")
print("1×1 卷积: 跨通道线性组合, 用来降/升通道 → 在大卷积前降维, 大幅省计算(GoogLeNet/ResNet bottleneck)\n")

# 一个简化的 Inception 模块: 多分支并行再拼接 / a simplified Inception module
class Inception(nn.Module):
    def __init__(self, cin):
        super().__init__()
        self.b1 = nn.Conv2d(cin, 16, 1)                                   # 1×1 分支 / 1x1 branch
        self.b2 = nn.Sequential(nn.Conv2d(cin,16,1), nn.Conv2d(16,24,3,padding=1))  # 1×1→3×3
        self.b3 = nn.Sequential(nn.Conv2d(cin,16,1), nn.Conv2d(16,24,5,padding=2))  # 1×1→5×5
        self.b4 = nn.Sequential(nn.MaxPool2d(3,1,1), nn.Conv2d(cin,16,1))           # pool→1×1
    def forward(self, x):
        return torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)   # 通道维拼接 / concat channels

inc = Inception(32); out = inc(torch.randn(1, 32, 28, 28))
print(f"Inception 模块: 输入 32 通道 → 4 分支并行 → 拼接输出 {out.shape[1]} 通道 (16+24+24+16)")
print("思想: 不选单一卷积核, 多尺度并行让网络自己学; 1×1 控制计算量")


```
演进: LeNet(范式确立) → AlexNet(ReLU/Dropout/更深/GPU, 点燃DL) → VGG(只用3×3堆叠)
      → GoogLeNet(Inception多尺度+1×1降维) → ResNet(残差,下一课)
LeNet范式: (卷积+池化)×N → 全连接; 至今CNN基本骨架
VGG: 多个3×3 = 同感受野(2个3×3=1个5×5) + 更少参数 + 更多非线性(ReLU)
1×1卷积: 跨通道线性组合, 降/升维, 大卷积前降维省计算(bottleneck)
Inception: 1×1/3×3/5×5/pool 多分支并行再拼接, 让网络学多尺度
```

### 💡 面试速查 / Interview cheat-sheet
1. **多个3×3代替大核**: 同感受野 + 省参数 + 更多非线性(VGG)。
   Stack 3×3 vs large kernel: same receptive field + fewer params + more nonlinearity.
2. **1×1卷积**: 跨通道融合 + 降/升维 + 省计算(bottleneck)。
   1×1 conv: channel mixing + reduce/expand + cheaper (bottleneck).
3. **AlexNet 关键**: ReLU + Dropout + 更深 + GPU + 数据增强。
   AlexNet keys: ReLU + Dropout + deeper + GPU + augmentation.
4. **Inception**: 多尺度并行分支拼接。
   Inception: parallel multi-scale branches concatenated.
5. **LeNet范式**: (卷积+池化)×N → 全连接, 沿用至今。
   LeNet paradigm: (conv+pool)×N → FC, still used.

### 下一节 / Next
**10.4 ResNet 与跳连**——网络越深越好？其实直接堆深会**退化**(更差)。ResNet 的**残差连接**用一个巧妙的"捷径"解决它，让网络能堆到上百层。我们会复现退化现象并验证残差的威力。
**10.4 ResNet & Skip Connections** — is deeper always better? Naively stacking deeper actually **degrades** performance. ResNet's **residual connections** fix it with a clever shortcut, enabling 100+ layers. We'll reproduce the degradation and verify residuals' power.
